Celda 1: Importación de librerías y carga de datos limpios

In [1]:
import pandas as pd
import numpy as np
import os
from imblearn.over_sampling import RandomOverSampler

PATH_LIMPIO = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\dataset\dataset_limpio.xlsx"
PATH_MENSUAL = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\dataset\dataset_mensual.xlsx"
PATH_OUTPUT_DIR = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\augmented"

os.makedirs(PATH_OUTPUT_DIR, exist_ok=True)

df_registro = pd.read_excel(PATH_LIMPIO)
df_mensual = pd.read_excel(PATH_MENSUAL)

print(f"Dataset registro: {len(df_registro)} registros")
print(f"Dataset mensual: {len(df_mensual)} meses")

Dataset registro: 340 registros
Dataset mensual: 289 meses


Celda 2: Bootstrapping Temporal sobre dataset_limpio

In [2]:
def bootstrap_temporal(df, n_iter=5, ruido_monto=50, ruido_dias=3):
    df_bootstrap = df.copy()
    df_bootstrap['origen'] = 'original'

    for i in range(n_iter):
        df_synthetic = df.sample(n=len(df), replace=True, random_state=42+i).copy()
        df_synthetic['Monto'] = df_synthetic['Monto'] + np.random.normal(0, ruido_monto, len(df_synthetic))
        df_synthetic['Monto'] = df_synthetic['Monto'].clip(lower=0)
        df_synthetic['Fecha'] = df_synthetic['Fecha'] + pd.to_timedelta(np.random.randint(-ruido_dias, ruido_dias+1, len(df_synthetic)), unit='D')
        df_synthetic['origen'] = f'bootstrap_{i+1}'
        df_bootstrap = pd.concat([df_bootstrap, df_synthetic], ignore_index=True)

    return df_bootstrap

df_bootstrap = bootstrap_temporal(df_registro)

print(f"Bootstrapping completado: {len(df_registro)} -> {len(df_bootstrap)} registros")
print(f"Distribución de origen:")
print(df_bootstrap['origen'].value_counts())

Bootstrapping completado: 340 -> 2040 registros
Distribución de origen:
origen
original       340
bootstrap_1    340
bootstrap_2    340
bootstrap_3    340
bootstrap_4    340
bootstrap_5    340
Name: count, dtype: int64


Celda 3: SMOTE (COMENTADO - No funciona con dataset pequeño, se usa RandomOverSampler en Celda 3.5)

In [3]:
# from imblearn.over_sampling import SMOTE
# from sklearn.preprocessing import LabelEncoder
#
# le_modelo = LabelEncoder()
# df_reg_smote = df_registro.copy()
# df_reg_smote['Ataud_Modelo_Enc'] = le_modelo.fit_transform(df_reg_smote['Ataud_Modelo'])
#
# X = df_reg_smote[['Monto', 'Carroza', 'Auto', 'Cargadores']]
# y = df_reg_smote['Ataud_Modelo_Enc']
#
# smote = SMOTE(sampling_strategy='auto', k_neighbors=1, random_state=42)
# X_res, y_res = smote.fit_resample(X, y)
#
# df_registro_aug = pd.DataFrame(X_res, columns=['Monto', 'Carroza', 'Auto', 'Cargadores'])
# df_registro_aug['Ataud_Modelo'] = le_modelo.inverse_transform(y_res)
#
# print(f"Dataset de registros aumentado con SMOTE: {len(df_registro_aug)} filas.")

Celda 3.5: Balanceo de Categorías con RandomOverSampler

In [4]:
def filtrar_clases_raros(df, min_registros=5):
    conteos = df['Ataud_Modelo'].value_counts()
    clases_validas = conteos[conteos >= min_registros].index.tolist()
    n_antes = df['Ataud_Modelo'].nunique()
    df_filtrado = df[df['Ataud_Modelo'].isin(clases_validas)].copy()
    n_despues = df_filtrado['Ataud_Modelo'].nunique()
    print(f"Filtrado: {n_antes} -> {n_despues} clases (se eliminaron {n_antes - n_despues} clases con <{min_registros} registros)")
    return df_filtrado

df_registro_filtrado = filtrar_clases_raros(df_registro, min_registros=5)

def aplicar_oversampling(df):
    df_bal = df.copy()
    features = ['Monto', 'Carroza', 'Auto', 'Cargadores', 'Ataud_Color', 'Capilla', 'Forma de pago', 'Ataud_completo']
    features_existentes = [f for f in features if f in df_bal.columns]

    X = df_bal[features_existentes]
    y = df_bal['Ataud_Modelo']

    ros = RandomOverSampler(random_state=42)
    X_res, y_res = ros.fit_resample(X, y)

    df_result = pd.DataFrame(X_res)
    df_result['Ataud_Modelo'] = y_res
    df_result['origen'] = 'oversample_sintetico'
    df_result.loc[df_result.index < len(df), 'origen'] = 'original'

    return df_result

df_oversample = aplicar_oversampling(df_registro_filtrado)

print(f"\nOversampling completado: {len(df_registro_filtrado)} -> {len(df_oversample)} registros")
print(f"Distribución de clases después:")
print(df_oversample['Ataud_Modelo'].value_counts())

Filtrado: 86 -> 11 clases (se eliminaron 75 clases con <5 registros)

Oversampling completado: 254 -> 836 registros
Distribución de clases después:
Ataud_Modelo
Imperial     76
sin_ataud    76
Americano    76
Madera       76
Principe     76
Biblia       76
Parvulo      76
Lincoln      76
Redondo      76
Modelo       76
Diamante     76
Name: count, dtype: int64


Celda 4: Sliding Window sobre dataset_mensual

In [5]:
def crear_sliding_windows(df_mensual, window_size=3, targets=None):
    if targets is None:
        targets = ['servicios_totales', 'monto_total']

    df_win = df_mensual.copy()
    if 'Periodo' in df_win.columns:
        df_win = df_win.set_index('Periodo')

    windows = []
    for i in range(window_size, len(df_win)):
        window = {}
        for target in targets:
            if target in df_win.columns:
                for lag in range(1, window_size+1):
                    window[f'{target}_lag_{lag}'] = df_win[target].iloc[i-lag]
                window[f'{target}_target'] = df_win[target].iloc[i]
        window['mes_target'] = str(df_win.index[i])
        windows.append(window)

    return pd.DataFrame(windows)

targets_disponibles = [c for c in ['servicios_totales', 'monto_total'] if c in df_mensual.columns]
df_windows = crear_sliding_windows(df_mensual, window_size=3, targets=targets_disponibles)

print(f"Sliding windows generadas: {len(df_windows)}")
print(f"Columnas: {list(df_windows.columns)}")

Sliding windows generadas: 286
Columnas: ['servicios_totales_lag_1', 'servicios_totales_lag_2', 'servicios_totales_lag_3', 'servicios_totales_target', 'monto_total_lag_1', 'monto_total_lag_2', 'monto_total_lag_3', 'monto_total_target', 'mes_target']


Celda 5: Reporte de Augmentation

In [6]:
def generar_reporte(df_orig, df_oversample, df_bootstrap, df_windows):
    reporte = []
    reporte.append(('registros_originales', len(df_orig)))
    reporte.append(('registros_oversample_total', len(df_oversample)))
    reporte.append(('registros_oversample_sinteticos', len(df_oversample) - len(df_orig)))
    reporte.append(('distribucion_clases_antes', str(df_orig['Ataud_Modelo'].value_counts().to_dict())))
    reporte.append(('distribucion_clases_despues', str(df_oversample['Ataud_Modelo'].value_counts().to_dict())))
    reporte.append(('registros_bootstrap_total', len(df_bootstrap)))
    reporte.append(('ventanas_generadas', len(df_windows)))
    reporte.append(('window_size', 3))
    return pd.DataFrame(reporte, columns=['Metrica', 'Valor'])

reporte = generar_reporte(df_registro, df_oversample, df_bootstrap, df_windows)
print("Reporte de augmentation:")
print(reporte.to_string(index=False))

Reporte de augmentation:
                        Metrica                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

Celda 6: Exportación de datasets

In [7]:
df_oversample.to_excel(os.path.join(PATH_OUTPUT_DIR, 'dataset_smote.xlsx'), index=False)
df_bootstrap.to_excel(os.path.join(PATH_OUTPUT_DIR, 'dataset_bootstrap.xlsx'), index=False)
df_windows.to_excel(os.path.join(PATH_OUTPUT_DIR, 'dataset_sliding_window.xlsx'), index=False)
reporte.to_excel(os.path.join(PATH_OUTPUT_DIR, 'reporte_augmentation.xlsx'), index=False)

print(f"Archivos exportados en: {PATH_OUTPUT_DIR}")
print("Pipeline de augmentation completado.")

Archivos exportados en: C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\augmented
Pipeline de augmentation completado.
